# Atomic Absorption Spectroscopy (AAS) Quantitative Analysis & Quality Control

This interactive notebook provides end-to-end data processing for Atomic Absorption Spectroscopy (FAAS & GFAAS):
1. **Raw Data Ingestion & Metadata Inspection** (CSV, TSV, XLSX)
2. **Calibration Curve Modeling** (Linear OLS, WLS, Zero-Intercept, Curvature Warnings)
3. **Analytical Quality Control (QC)** Compliance Verification (US EPA Method 7000B & 7010)
4. **Quantitative Sample Concentration & Below-LOD/LOQ Flagging**
5. **Method of Standard Additions (MOSA)** for Matrix-Suppressed Samples with Delta-Method Uncertainty
6. **Catalyst Metal Loading Quantification (wt.%)** from Acid Digestions
7. **Electrochemical Dissolution & Leaching Kinetics** with Cumulative Aliquot Tracking
8. **Journal-Quality Multi-Format Figure Exports** with Cryptographic Provenance Tracking.

In [1]:
import sys
from pathlib import Path
root_path = Path.cwd().parent.parent if Path.cwd().name == 'characterization' else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from parsers.characterization.aas import parse_aas_file
from modules.characterization.aas import (
    fit_aas_calibration,
    fit_standard_addition,
    quantify_aas_samples,
    evaluate_aas_qc,
    calculate_aas_catalyst_loading,
    calculate_aas_leaching,
    plot_aas_calibration,
    plot_standard_addition,
    plot_aas_concentration_comparison,
    plot_aas_catalyst_loading,
    plot_aas_leaching,
    plot_aas_stability_before_after,
    plot_aas_replicate_distribution,
    plot_aas_qc_chart,
)
from parsers.characterization.common import record_characterization_provenance
from scripts.utils import apply_publication_style

apply_publication_style()
print('AAS processing and visualization modules loaded successfully.')

AAS processing and visualization modules loaded successfully.


## 1. Raw Data Ingestion & Metadata Inspection

Load Flame AAS (FAAS) Fe calibration standards and inspect instrument metadata.

In [2]:
base_data = root_path / 'data' / 'examples' / 'characterization' / 'aas'
faas_data = parse_aas_file(base_data / 'faas_calibration_fe.csv', technique='FAAS')
print('Technique:', faas_data.technique)
print('Instrument Model:', faas_data.metadata.get('instrument_model'))
print('Background Correction:', faas_data.metadata.get('background_correction'))
print('Elements Detected:', faas_data.metadata.get('elements_detected'))
faas_df = faas_data.dataframe
faas_df.head(10)

Technique: FAAS
Instrument Model: Unknown AAS Spectrometer
Background Correction: Unspecified
Elements Detected: ['Fe']


,sample_id,element,sample_type,replicate,absorbance,integrated_absorbance,concentration_reported,dilution_factor,wavelength_nm,unit
0,BLANK_CAL,Fe,Blank,1,0.0022,NaN,0.0,1.0,248.3,mg/L
1,BLANK_CAL,Fe,Blank,2,0.0017,NaN,0.0,1.0,248.3,mg/L
2,BLANK_CAL,Fe,Blank,3,0.0023,NaN,0.0,1.0,248.3,mg/L
3,STD_0.20,Fe,Standard,1,0.0195,NaN,0.2,1.0,248.3,mg/L
4,STD_0.20,Fe,Standard,2,0.0181,NaN,0.2,1.0,248.3,mg/L
5,STD_0.20,Fe,Standard,3,0.0181,NaN,0.2,1.0,248.3,mg/L
6,STD_0.50,Fe,Standard,1,0.0443,NaN,0.5,1.0,248.3,mg/L
7,STD_0.50,Fe,Standard,2,0.0437,NaN,0.5,1.0,248.3,mg/L
8,STD_0.50,Fe,Standard,3,0.0427,NaN,0.5,1.0,248.3,mg/L
9,STD_1.00,Fe,Standard,1,0.0847,NaN,1.0,1.0,248.3,mg/L


## 2. Calibration Curve Modeling & Detection Limits

Perform linear regression ($A = mC + b$), derive sensitivity slope, $R^2$, and IUPAC detection limit ($3.3 s_b / m$).

In [3]:
stds_df = faas_df[faas_df['sample_type'] == 'Standard']
blanks_df = faas_df[faas_df['sample_type'] == 'Blank']
cal_fe = fit_aas_calibration(
    standards_concentration=stds_df['concentration_reported'].values,
    standards_signal=stds_df['absorbance'].values,
    signal_type='absorbance',
    blank_signals=blanks_df['absorbance'].values,
    lod_method='blank_stdev',
)
print(f"Slope: {cal_fe['slope']:.4f} AU/(mg/L)")
print(f"Intercept: {cal_fe['intercept']:.4f} AU")
print(f"R^2: {cal_fe['r_squared']:.5f}")
print(f"LOD: {cal_fe['lod_conc']:.4f} mg/L, LOQ: {cal_fe['loq_conc']:.4f} mg/L")

fig_cal, (ax_top, ax_bot) = plot_aas_calibration(cal_fe, element='Iron (Fe 248.3 nm)', unit='mg/L')
plt.show()

Slope: 0.0824 AU/(mg/L)
Intercept: 0.0019 AU
R^2: 0.99997
LOD: 0.0129 mg/L, LOQ: 0.0390 mg/L


C:\Users\mjram\python_graphing\modules\characterization\aas.py:744: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
C:\Users\mjram\python_graphing\modules\characterization\aas.py:744: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_6292\3819645621.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Analytical Quality Control (QC) Compliance (US EPA 7000B)

Verify calibration linearity, blank cleanliness, ICV recovery, and replicate precision against EPA Method 7000B.

In [4]:
qc_report = evaluate_aas_qc(faas_df, cal_fe, sop='EPA_7000B')
print('Overall QC Status:', qc_report['overall_status'])
display(qc_report['checks'])

qc_run_df = pd.read_csv(base_data / 'aas_qc_run_sequence.csv')
fig_qc, ax_qc = plot_aas_qc_chart(qc_run_df['QC_Check'], qc_run_df['Recovery_pct'], sop='EPA_7000B')
plt.show()

Overall QC Status: PASS


,check,value,criteria,status
0,Calibration Linearity (R^2),0.99997,>= 0.995,PASS
1,Calibration / Method Blank,0.0016 (Sig: 0.0021),<= LOD (0.0129),PASS
2,ICV Recovery (ICV_1.0mgL),99.8%,90.0-110.0%,PASS
3,Replicate Precision (%RSD),All within limit,RSD <= 10.0%,PASS


C:\Users\mjram\python_graphing\modules\characterization\aas.py:1122: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_6292\4089356514.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Method of Standard Additions (MOSA) for Matrix Suppression

When analyzing complex high-salt electrolyte samples, matrix interference suppresses analyte sensitivity.
Standard addition fits $S = m \Delta C + S_0$ and extrapolates backward to find sample concentration.

In [5]:
mosa_data = parse_aas_file(base_data / 'aas_standard_addition_catalyst.csv')
mosa_df = mosa_data.dataframe
spikes = [0.0, 0.50, 1.00, 1.50, 2.00]
sigs = [mosa_df[mosa_df['sample_id'].str.contains(f'\\+{sp:.2f}') if sp > 0 else mosa_df['sample_id'].str.contains('Unspiked')]['absorbance'].mean() for sp in spikes]

sa_res = fit_standard_addition(spikes, sigs, sample_volume_mL=5.0, final_volume_mL=25.0)
print(f"Apparent Flask Conc: {sa_res['apparent_concentration']:.3f} +/- {sa_res['ci95_apparent']:.3f} mg/L")
print(f"True Original Sample Conc: {sa_res['original_concentration']:.3f} +/- {sa_res['ci95_original']:.3f} mg/L (DF = {sa_res['dilution_factor']:.1f})")

fig_sa, ax_sa = plot_standard_addition(sa_res, element='Cobalt (Co in 0.1 M KOH)', unit='mg/L')
plt.show()

C:\Users\mjram\python_graphing\modules\characterization\aas.py:812: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


Apparent Flask Conc: 0.841 +/- 0.025 mg/L
True Original Sample Conc: 4.204 +/- 0.125 mg/L (DF = 5.0)


C:\Users\mjram\AppData\Local\Temp\ipykernel_6292\4154139973.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Catalyst Composition & Metal Loading (wt.%)

Quantify metal loading from acid-digested solid catalyst samples and compare with nominal targets.

In [6]:
dig_df = pd.read_csv(base_data / 'catalyst_digestion_loading.csv')
samples = dig_df[dig_df['Sample_Type'] == 'Sample']
pt_samples = samples[samples['Element'] == 'Pt']
loadings = [calculate_aas_catalyst_loading(r['Measured_Conc_mg_L'], r['Digest_Vol_mL'], r['Mass_mg'], r['Dilution_Factor'])['metal_loading_wt_pct'] for _, r in pt_samples.iterrows()]
print(f"Pt/C Actual Loading: {np.mean(loadings[:3]):.2f} +/- {np.std(loadings[:3], ddof=1):.2f} wt% (Nominal: 20.0 wt%)")

fig_load, ax_load = plot_aas_catalyst_loading(
    ['Pt/C', 'PtCo/C (Pt)', 'PtCo/C (Co)', 'NiFe (Ni)', 'NiFe (Fe)'],
    [19.8, 14.8, 5.1, 24.5, 12.2],
    nominal_loading_wt_pct=[20.0, 15.0, 5.0, 25.0, 12.0],
    error_bars=[0.2, 0.2, 0.1, 0.3, 0.2],
)
plt.show()

Pt/C Actual Loading: 19.79 +/- 0.02 wt% (Nominal: 20.0 wt%)


C:\Users\mjram\python_graphing\modules\characterization\aas.py:948: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_6292\1425886125.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Electrocatalytic Dissolution & Leaching Kinetics

Track metal dissolution during a 100-hour alkaline OER stability test with aliquot sampling corrections.

In [7]:
leach_df = pd.read_csv(base_data / 'catalyst_stability_leaching.csv')
co_df = leach_df[leach_df['Element'] == 'Co']
time_pts = co_df['Time_hours'].tolist()
pcts = []
losses = []
aliquots = []
for _, r in co_df.iterrows():
    res = calculate_aas_leaching(
        r['Measured_Conc_mg_L'], r['Electrolyte_Volume_mL'], r['Initial_Catalyst_Mass_mg'],
        r['Initial_Metal_wt_pct'] / 100.0, aliquot_history=aliquots if aliquots else None
    )
    pcts.append(res['percentage_metal_leached'])
    losses.append(res['mass_normalized_leaching_ug_per_mg'])
    if r['Aliquot_Withdrawn_mL'] > 0:
        aliquots.append((r['Aliquot_Withdrawn_mL'], r['Measured_Conc_mg_L']))

print(f"Total Co Leached after 100h: {pcts[-1]:.2f}%")
print(f"Mass-normalized metal loss: {losses[-1]:.2f} ug/mg_cat")

fig_leach, ax_leach = plot_aas_leaching([f'{t}h' for t in time_pts], pcts, mass_normalized_loss_ug_per_mg=losses, element='Cobalt')
plt.show()

Total Co Leached after 100h: 21.34%
Mass-normalized metal loss: 10.88 ug/mg_cat


C:\Users\mjram\python_graphing\modules\characterization\aas.py:993: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_6292\3116722670.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
